In [12]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [39]:
# 1. Data processing - Trade
df_trade = pd.read_csv("TradeData_12_11_2025_3_13_25.csv", encoding="cp1252")
df_trade = df_trade[df_trade['reporterCode'] == 'VNM']
df_trade = df_trade[['refPeriodId', 'partnerCode', 'cifvalue']]
df_trade.columns = ['year', 'iso3', 'export']
df_trade

,year,iso3,export
C,2014,W00,1.502171e+11
C,2014,DZA,2.463514e+08
C,2014,AND,3.520000e+02
C,2014,AGO,8.143160e+07
C,2014,AZE,7.359021e+07
...,...,...,...
C,2022,USA,1.094596e+11
C,2022,URY,1.031145e+08
C,2022,UZB,3.646101e+07
C,2022,VEN,5.688213e+07


In [52]:
# 2. GDP DATA (Sửa lỗi 4 dòng tiêu đề thừa)
# skiprows=4 là chìa khóa để đọc file này
df_gdp = pd.read_csv("GDP_PolicyNote2.csv", skiprows=4)
# File này đang ở dạng ngang (Wide), cần chuyển sang dọc (Long) để ghép
year_cols = [c for c in df_gdp.columns if c.isdigit()]

df_gdp1 = df_gdp.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='gdp'
)

df_gdp1 = df_gdp1[['Country Code', 'year', 'gdp']]
df_gdp1.columns = ['iso3', 'year', 'gdp']
df_gdp1['year'] = pd.to_numeric(df_gdp1['year'], errors='coerce') # Chuyển năm sang số

df_gdp1

,iso3,year,gdp
0,ABW,1960,NaN
1,AFE,1960,2.420993e+10
2,AFG,1960,NaN
3,AFW,1960,1.190511e+10
4,AGO,1960,NaN
...,...,...,...
17285,XKX,2024,1.114860e+10
17286,YEM,2024,NaN
17287,ZAF,2024,4.002607e+11
17288,ZMB,2024,2.632578e+10


In [55]:
# 3. DISTANCE (Tự tính từ file geo_cepii)
# Vì bạn tải nhầm file geo (chỉ có tọa độ) thay vì dist (có khoảng cách)
# Nên ta sẽ tự tính khoảng cách "đường chim bay" (Great Circle Distance)
df_geo = pd.read_csv("geo_cepii(geo_cepii).csv", encoding="cp1252")
# Lấy tọa độ Việt Nam
vnm_coords = df_geo[df_geo['iso3'] == 'VNM'].iloc[0]
lat1, lon1 = np.radians(vnm_coords['lat']), np.radians(vnm_coords['lon'])

# Tính cho các nước khác
df_geo['lat_rad'] = np.radians(df_geo['lat'])
df_geo['lon_rad'] = np.radians(df_geo['lon'])
# Công thức Haversine
dlon = df_geo['lon_rad'] - lon1
dlat = df_geo['lat_rad'] - lat1
a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(df_geo['lat_rad']) * np.sin(dlon/2)**2
c = 2 * np.arcsin(np.sqrt(a))
R = 6371 # Bán kính trái đất (km)
df_geo['distance'] = c * R
df_dist = df_geo[['iso3', 'distance']]

df_dist

,iso3,distance
0,ABW,16256.719853
1,AFG,3880.467128
2,AGO,10627.684952
3,AIA,15489.002807
4,ALB,8165.332036
...,...,...
233,ZAF,9853.288632
234,ZAF,11064.037575
235,ZAR,10237.567394
236,ZMB,9385.065711


In [66]:
# 4. LPI DATA (Gộp các năm lại)
import pandas as pd

# Danh sách file (đúng tên bạn đã upload)
lpi_files = {
    2014: "International_LPI_from_2007_to_2023_0.xlsx - 2014.csv",
    2016: "International_LPI_from_2007_to_2023_0.xlsx - 2016.csv",
    2018: "International_LPI_from_2007_to_2023_0.xlsx - 2018.csv",
    2023: "International_LPI_from_2007_to_2023_0.xlsx - 2023.csv"
}

lpi_list = []
# 1. Tạo từ điển map Tên -> Mã (Dùng năm 2018 làm chuẩn vì nó có cả hai)
name_to_code = {}

# --- BƯỚC 1: ĐỌC CÁC NĂM CŨ (2014, 2016, 2018) ---
for year in [2014, 2016, 2018]:
    file_path = lpi_files[year]
    try:
        # Các năm này header nằm ở dòng 3 (index 2)
        temp = pd.read_csv(file_path, header=2)

        # Tìm cột Code và Score
        col_code = [c for c in temp.columns if "Code" in str(c)][0]
        col_score = [c for c in temp.columns if "Score" in str(c) or "score" in str(c)][0]
        col_name = [c for c in temp.columns if "Country" in str(c)][0] # Cột tên nước

        # Lưu vào từ điển mapping (để dành cho năm 2023 dùng)
        for index, row in temp.iterrows():
            if pd.notna(row[col_name]) and pd.notna(row[col_code]):
                name_to_code[row[col_name]] = row[col_code]

        # Lấy dữ liệu
        temp = temp[[col_code, col_score]].copy()
        temp.columns = ['iso3', 'lpi_score']
        temp['year'] = year
        lpi_list.append(temp)
        print(f"✅ Đã đọc xong năm {year}")
    except Exception as e:
        print(f"❌ Lỗi năm {year}: {e}")

# --- BƯỚC 2: ĐỌC NĂM 2023 (CẤU TRÚC ĐẶC BIỆT) ---
try:
    # Năm 2023 header nằm ngay dòng đầu (header=0)
    df_2023 = pd.read_csv(lpi_files[2023], header=0)

    # Đổi tên cột cho dễ xử lý
    # File 2023 cột tên nước là "Economy", điểm là "LPI Score"
    df_2023 = df_2023.rename(columns={"Economy": "Country Name", "LPI Score": "lpi_score"})

    # TỰ ĐỘNG ĐIỀN MÃ NƯỚC (ISO3) từ từ điển đã tạo ở Bước 1
    df_2023['iso3'] = df_2023['Country Name'].map(name_to_code)

    # Chỉ giữ lại dòng nào map được mã (bỏ các nước lạ hoặc dòng trống)
    df_2023 = df_2023.dropna(subset=['iso3'])

    df_2023 = df_2023[['iso3', 'lpi_score']]
    df_2023['year'] = 2023

    lpi_list.append(df_2023)
    print(f"✅ Đã đọc xong năm 2023 (Tự động điền mã ISO cho {len(df_2023)} nước)")
except Exception as e:
    print(f"❌ Lỗi năm 2023: {e}")

# --- BƯỚC 3: GỘP LẠI ---
if lpi_list:
    df_lpi = pd.concat(lpi_list, ignore_index=True)
    print("\n🎉 THÀNH CÔNG! Kết quả:")
    print(df_lpi.groupby('year').size()) # Kiểm tra số lượng dòng mỗi năm
    print(df_lpi.tail())
else:
    print("Không có dữ liệu nào được gộp.")

✅ Đã đọc xong năm 2014
✅ Đã đọc xong năm 2016
✅ Đã đọc xong năm 2018
✅ Đã đọc xong năm 2023 (Tự động điền mã ISO cho 135 nước)

🎉 THÀNH CÔNG! Kết quả:
year
2014    160
2016    160
2018    160
2023    135
dtype: int64
    iso3  lpi_score  year
610  CMR        2.1  2023
611  HTI        2.1  2023
612  SOM        2.0  2023
613  AFG        1.9  2023
614  LBY        1.9  2023


In [83]:
df_lpi

,iso3,lpi_score,year
0,DEU,4.12,2014
1,NLD,4.05,2014
2,BEL,4.04,2014
3,GBR,4.01,2014
4,SGP,4.00,2014
...,...,...,...
610,CMR,2.10,2023
611,HTI,2.10,2023
612,SOM,2.00,2023
613,AFG,1.90,2023


In [82]:
# --- BƯỚC 2: GHÉP TẤT CẢ LẠI (MERGE) ---

# Chuẩn hóa mã nước
for df in [df_trade, df_gdp1, df_dist, df_lpi]:
    df['iso3'] = df['iso3'].astype(str).str.strip().str.upper()

# Ghép lần lượt: Trade + LPI + GDP + Distance
data = pd.merge(df_trade, df_lpi, on=['iso3', 'year'], how='inner')
data = pd.merge(data, df_gdp1, on=['iso3', 'year'], how='inner')
data = pd.merge(data, df_dist, on=['iso3'], how='inner')

# Lọc dữ liệu sạch (Bỏ số 0 hoặc âm)
data = data[(data['export'] > 0) & (data['gdp'] > 0) & (data['distance'] > 0)]

print(f"Số lượng quan sát sau khi ghép: {len(data)}")

# --- BƯỚC 3: CHẠY MÔ HÌNH ---
if len(data) > 0:
    # Mô hình Gravity: ln(Export) = ln(GDP) + ln(Distance) + LPI
    # MÔ HÌNH CŨ (Basic Gravity):
    # model = smf.ols("np.log(export) ~ np.log(gdp) + np.log(distance) + lpi_score", data=model_data).fit()

    # MÔ HÌNH MỚI (Advanced - Có Time Fixed Effects):
    # Thêm cụm " + C(year) " vào cuối công thức
    model_fe = smf.ols("np.log(export) ~ np.log(gdp) + np.log(distance) + lpi_score + C(year)", data=data).fit() # Changed gdp1 to gdp and model_data to data

    print(model_fe.summary())
else:
    print("Không đủ dữ liệu để chạy. Kiểm tra lại mã nước hoặc năm.")

Số lượng quan sát sau khi ghép: 382
                            OLS Regression Results                            
Dep. Variable:         np.log(export)   R-squared:                       0.761
Model:                            OLS   Adj. R-squared:                  0.758
Method:                 Least Squares   F-statistic:                     239.2
Date:                Thu, 11 Dec 2025   Prob (F-statistic):          2.14e-114
Time:                        10:18:51   Log-Likelihood:                -604.38
No. Observations:                 382   AIC:                             1221.
Df Residuals:                     376   BIC:                             1244.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Inte

/tmp/ipython-input-1555099459.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['iso3'] = df['iso3'].astype(str).str.strip().str.upper()


In [76]:
print("Descriptive statistics for variables used in the model:")
print(data[['export', 'gdp', 'distance', 'lpi_score']].describe())

Descriptive statistics for variables used in the model:
             export           gdp      distance   lpi_score
count  3.820000e+02  3.820000e+02    382.000000  382.000000
mean   1.881702e+09  8.508514e+11   9028.018226    3.049267
std    5.761481e+09  2.660475e+12   4422.345873    0.589877
min    2.481000e+03  1.135251e+09    478.109869    1.600000
25%    3.154390e+07  3.298174e+10   6905.935643    2.580000
50%    1.749062e+08  1.194570e+11   8444.878251    2.970000
75%    1.332353e+09  4.493839e+11  11252.104760    3.527500
max    4.758011e+10  2.065652e+13  19210.982853    4.230000
